In [81]:
import numpy as np
import pandas as pd
import tensorflow as tf

import sys
import pickle

sys.path.append('..')
sys.path.append('../..')

from src.Environment.environment_v2 import MyModelSelectionEnv
from src.utils import train_test_anomaly
from src.Components.data_processing import data_process
from sklearn.model_selection import train_test_split

In [82]:
import tf_agents.bandits.agents as bandit_agents
from tf_agents.metrics import tf_metrics
from tf_agents.trajectories import time_step as ts
from tf_agents.drivers import dynamic_step_driver
from tf_agents.environments import tf_py_environment
from tf_agents.networks import q_network, NestFlatten
from tf_agents.bandits.replay_buffers import bandit_replay_buffer
from tf_agents.metrics import export_utils

### Importing Data and Setting Up the Environment

In [83]:
file_path = '../datasets/Dodgers/101-freeway-traffic.test.out'

columns = ['value', 'anomaly']

df = pd.read_csv(file_path, names=columns, header=None)

In [84]:
# _, test_data = train_test_anomaly(df)

train_data, test_data = train_test_split(df, test_size=0.3, shuffle=False)


list_threshold = [0.00757, 1.5832, 2.88065]
list_gtruth = test_data[['anomaly']].values.reshape(-1, 1)

In [85]:
train_data = train_data[train_data['value'] >= 0]

In [86]:
test_np = test_data['value'].values.reshape(-1, 1)

In [87]:
train_env = MyModelSelectionEnv(train_data, list_thresholds=list_threshold, list_gtruth=list_gtruth)
train_environment = tf_py_environment.TFPyEnvironment(train_env) # Converts the PyEnvironment to TFEnvironment

### Setting Up the Neural Epsilon Greedy Agent

In [88]:
action_spec = train_environment.action_spec()
observation_spec = train_environment.time_step_spec().observation

EPSILON = 0.15
LAYERS = (20, 20, 20)
LR = 0.005

TRAINING_LOOPS = 500
steps_per_loop = 1
async_steps_per_loop = 1

network = q_network.QNetwork(
          input_tensor_spec=observation_spec,
          action_spec=action_spec,
          fc_layer_params=LAYERS,
      )

In [89]:

eps_agent = bandit_agents.neural_epsilon_greedy_agent.NeuralEpsilonGreedyAgent(action_spec=train_environment.action_spec(), time_step_spec=train_environment.time_step_spec(), reward_network=network, optimizer=tf.compat.v1.train.AdamOptimizer(learning_rate=LR),
        epsilon=EPSILON,
        emit_policy_info='predicted_rewards_mean',
        info_fields_to_inherit_from_greedy=['predicted_rewards_mean'])

eps_agent.initialize()

### Replay Buffers and Drivers

In [90]:
data_spec = eps_agent.policy.trajectory_spec

In [91]:
def get_replay_buffer(
    data_spec, batch_size, steps_per_loop, async_steps_per_loop
):
  """Return a `TFUniformReplayBuffer` for the given `agent`."""
  return bandit_replay_buffer.BanditReplayBuffer(
      data_spec=data_spec,
      batch_size=batch_size,
      max_length=steps_per_loop * async_steps_per_loop,
  )

In [92]:
replay_buffer = get_replay_buffer(
      data_spec, train_environment.batch_size, steps_per_loop, async_steps_per_loop
  )

In [93]:
# Observers

step_metric = tf_metrics.EnvironmentSteps()
metrics = [
      tf_metrics.NumberOfEpisodes(), # Counts the number of episodes in the environment
      tf_metrics.AverageEpisodeLengthMetric(batch_size=train_environment.batch_size),   # Metric to compute the average episode length
      tf_metrics.AverageReturnMetric(batch_size=train_environment.batch_size) # Metric to compute the average return
  ]

add_batch_fn = replay_buffer.add_batch # Adds a batch of items on the replay buffer

observers = [add_batch_fn, step_metric] + metrics # List of observers for the driver

In [94]:
driver = dynamic_step_driver.DynamicStepDriver(
      env=train_environment,
      policy=eps_agent.collect_policy,
      num_steps=steps_per_loop * train_environment.batch_size,
      observers=observers,
  )

In [95]:
# Replay Buffer Values

"""
    The .as_dataset method creates and returns a dataset as entries from the buffer.

    A single entry from the dataset is the result of the following pipeline:

    - Sample sequences from the underlying data store
    - (Optional) Process them with 'sequence_preprocess_fn'
    - (Optional) Split them into subsequences of length num_steps
    - (Optional) Batch them into batches of size 'sample_batch_size'

    In practice, this pipeline is executed in parallel as much as possible if num_parallel_calls != 1.

    
"""

dataset_it = iter(
        replay_buffer.as_dataset(
            sample_batch_size=1,
            num_steps=1,
            single_deterministic_pass=True,
        )
    )

In [96]:
def training_loop(train_step, metrics):
 
    driver.run()
     
    batch_size = driver.env.batch_size
    dataset_it = iter(
        replay_buffer.as_dataset(
            sample_batch_size=batch_size,
            num_steps=1,
            single_deterministic_pass=True,
        )
    )

    meter = driver.observers[1:]
    for batch_id in range(async_steps_per_loop):
      experience, unused_buffer_info = dataset_it.get_next()
      loss_info = eps_agent.train(experience)
      print(loss_info)

      export_utils.export_metrics(
          step=train_step * async_steps_per_loop + batch_id,
          metrics=meter,
          loss_info=loss_info,
      )

    replay_buffer.clear()


In [ ]:
for i in range(1, 1000):

    training_loop(train_step=i, metrics=metrics)

In [98]:
policy = eps_agent.policy

### Evaluating the policy

In [99]:
test_data = test_data.reset_index()

In [100]:
test_env = MyModelSelectionEnv(test_data, list_thresholds=list_threshold, list_gtruth=list_gtruth)
test_environment = tf_py_environment.TFPyEnvironment(test_env) # Converts the PyEnvironment to TFEnvironment

In [101]:
test_data['anomaly'].iloc[51:1051].value_counts()

anomaly
0    857
1    143
Name: count, dtype: int64

In [ ]:
# Trained Policy Steps in Environment

act = 0
policy_state = policy.get_initial_state(batch_size=1)
act_list = []
score_list = []
subseq = test_env.subsequences

aa = test_environment.reset()

for _ in range(1000):

    time_step = test_environment.step(act)
    policy_return = policy.action(time_step, policy_state)
    act = policy_return.action.numpy()
    _, scr = test_env._apply_action(act)
    score_list.append(scr)
    print(test_env.pointer)
    policy_state = policy_return.state
    act_list.append(act)

In [103]:
act_list

[array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([0], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([0], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([1], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([0], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([2], dtype=int32),
 array([0], dtype=int32),
 array([2], dtype=int32),
 array([2], 

In [104]:
sample_gtruth = test_data['anomaly'].iloc[51:1051]
sample_gtruth.value_counts()

anomaly
0    857
1    143
Name: count, dtype: int64

In [105]:
score_np = np.array(score_list)
uni_val, uni_counts = np.unique(score_np, return_counts=True)
print(uni_val, uni_counts)

[0 1] [830 170]


In [108]:
compare_df = pd.DataFrame({'Gtruth': sample_gtruth.values, 'Y_Pred': score_np})
compare_df['Equal'] = compare_df['Gtruth'] == compare_df['Y_Pred']
compare_df

,Gtruth,Y_Pred,Equal
0,0,0,True
1,0,0,True
2,0,0,True
3,0,0,True
4,0,1,False
...,...,...,...
995,0,0,True
996,0,0,True
997,0,0,True
998,0,0,True


In [109]:
from sklearn.metrics import precision_score, recall_score, f1_score

prec = precision_score(sample_gtruth, score_np)
recall = recall_score(sample_gtruth, score_np)
f1 = f1_score(sample_gtruth, score_np)

print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precision Score: 0.1294  Recall: 0.1538  f1_score: 0.1406


In [ ]:
def load_models():
    
    model1 = pickle.load(open(f'../saved_models/iforest_dodgers_v2.sav','rb'))
    model2 = pickle.load(open(f'../saved_models/osvm_dodgers_v2.sav', 'rb'))
    model3 = pickle.load(open(f'../saved_models/copod_dodgers_v2.sav', 'rb'))
    model4 = pickle.load(open(f'../saved_models/clof_dodgers_v2.sav', 'rb')) 

    return [model1, model2, model3, model4]

In [ ]:
subseq[0]

In [ ]:
pred_vote_array = np.zeros((len(test_data), 2))
pred_vote_array[0, 1] = 1
pred_vote_array

In [ ]:
def voting_fn(anomaly_pred):

    pred = np.zeros((len(anomaly_pred), 2))

    for i, val in enumerate(anomaly_pred):

        if val == 0:

            pred[i, 0] += 1
        
        if val == 1:

            pred[i, 1] += 1
    
    return pred

In [ ]:
def eval_action(action_list, subsequence):

    models = load_models()
    pred_list = []
    pred_vote_array = np.zeros((len(test_data), 2))
    point = 0
    
    for i, acti in enumerate(action_list):

        if acti == 0:

            feats = subsequence[i].reshape(-1,1)
            score = models[0].decision_function(feats)

            label_list = [1 if i < list_threshold[0] else 0 for i in score]
            label_np = np.array(label_list)
            pred_list.append(label_np)
            voter = voting_fn(label_list)

            pred_vote_array[point:point+50] += voter

            
        
        elif acti == 1:

            feats = subsequence[i].reshape(-1,1)
            score = models[1].decision_function(feats)

            label_list = [1 if i < list_threshold[1] else 0 for i in score]
            label_np = np.array(label_list)
            pred_list.append(label_np)
            voter = voting_fn(label_list)
            pred_vote_array[point:point+50] += voter
        
        elif acti == 2:
           
           feats = subsequence[i].reshape(-1, 1)
           score = models[2].decision_function(feats)

           label_list = [1 if i > list_threshold[2] else 0 for i in score]
           label_np = np.array(label_list)
           voter = voting_fn(label_list)
           pred_vote_array[point:point+50] += voter
        
        elif acti == 3:
           
           feats = subsequence[i].reshape(-1, 1)
           score = models[3].decision_function(feats)

           label_list = [1 if i > list_threshold[3] else 0 for i in score]
           label_np = np.array(label_list)
           voter = voting_fn(label_list)
           pred_vote_array[point:point+50] += voter
        
        point += 1
    
    return pred_list, pred_vote_array

In [ ]:
predictions, vote_values = eval_action(act_list, subseq)

In [ ]:
vote_values[50]

In [ ]:
y_predyy = []

for rows in vote_values:

    if rows[0] > rows[1]:
        y_predyy.append(0)
    
    elif rows[0] < rows[1]:
        y_predyy.append(1)
    
    else:
        y_predyy.append(0)
    
y_predyy = np.array(y_predyy)
y_predyy

In [ ]:
thres_val, thres_count = np.unique(y_predyy, return_counts=True)
print(thres_val, thres_count)

In [ ]:
thres_np = np.where(vote_values[:, 1] > 1, 1, 0)
thres_np

In [ ]:
thres_val, thres_count = np.unique(thres_np, return_counts=True)
print(thres_val, thres_count)

In [ ]:
gtruth = test_data[['anomaly']]
test_data['anomaly'].value_counts()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

prec = precision_score(gtruth, thres_np)
recall = recall_score(gtruth, thres_np)
f1 = f1_score(gtruth, thres_np)

print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')